In [12]:
# loader scripts

import requests
import pandas as pd
import numpy as np

def to_tz(df_, time_col, tz_offset, tz_name):
    return (df_
    .groupby(tz_offset)
    [time_col]
    .transform(lambda s: pd.to_datetime(s)
    .dt.tz_localize(s.name, ambiguous=True)
    .dt.tz_convert(tz_name))
    )



def load_fuel_economy_data() -> pd.DataFrame :
    url = 'https://www.fueleconomy.gov/feg/epadata/vehicles.csv'
    cols = ['year', 'make', 'model', 'trany', 'drive', 'VClass', 'eng_dscr',
            'barrels08', 'city08', 'comb08', 'range', 'evMotor', 'cylinders', 'displ', 'fuelCost08',
            'fuelType', 'highway08', 'trans_dscr','createdOn']

    raw =  pd.read_csv(url)
    autos = (raw
        .loc[:, cols]
        .assign(
            # Extract Timezone Offset (e.g., EDT -> EST5EDT)
            offset=(raw.createdOn.str.extract(r'\d\d:\d\d (?P<offset>[A-Z]{3}?)')
                    ['offset'].replace('EDT', 'EST5EDT')),
            
            # Reconstruct date string (removing the original TZ name for parsing)
            str_date=(raw.createdOn.str.slice(4, 19) + " " +
                    raw.createdOn.str.slice(-4)),
            
            # Convert to localized datetime
            createdOn=lambda df_: to_tz(df_, 'str_date', 'offset', 'America/New_York')
        )
        .drop(columns=['offset', 'str_date']) # Clean up temp columns
        )
    return autos

In [21]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

C:\Users\dsgou\AppData\Local\Temp\ipykernel_27408\4282237246.py:24: DtypeWarning: Columns (69,71,72,73,74,75,77,80) have mixed types. Specify dtype option on import or set low_memory=False.
  raw =  pd.read_csv(url)


,year,make,model,trany,drive,VClass,eng_dscr,barrels08,city08,comb08,range,evMotor,cylinders,displ,fuelCost08,fuelType,highway08,trans_dscr,createdOn
0,1985,Alfa Romeo,Spider Veloce 2000,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(FFS),14.167143,19,21,0,NaN,4.0,2.0,2850,Regular,25,NaN,2013-01-01 00:00:00-05:00
1,1985,Ferrari,Testarossa,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(GUZZLER),27.046364,9,11,0,NaN,12.0,4.9,5450,Regular,14,NaN,2013-01-01 00:00:00-05:00
2,1985,Dodge,Charger,Manual 5-spd,Front-Wheel Drive,Subcompact Cars,(FFS),11.018889,23,27,0,NaN,4.0,2.2,2200,Regular,33,SIL,2013-01-01 00:00:00-05:00
3,1985,Dodge,B150/B250 Wagon 2WD,Automatic 3-spd,Rear-Wheel Drive,Vans,NaN,27.046364,10,11,0,NaN,8.0,5.2,5450,Regular,12,NaN,2013-01-01 00:00:00-05:00
4,1993,Subaru,Legacy AWD Turbo,Manual 5-spd,4-Wheel or All-Wheel Drive,Compact Cars,"(FFS,TRBO)",15.658421,17,19,0,NaN,4.0,2.2,3650,Premium,23,NaN,2013-01-01 00:00:00-05:00


In [22]:
df.VClass

0            Two Seaters
1            Two Seaters
2        Subcompact Cars
3                   Vans
4           Compact Cars
              ...       
49841       Compact Cars
49842       Compact Cars
49843       Compact Cars
49844       Compact Cars
49845       Compact Cars
Name: VClass, Length: 49846, dtype: object

In [23]:
# create a column per unique value of df.vClass
pd.get_dummies(df.VClass)

,Compact Cars,Large Cars,Midsize Cars,Midsize Station Wagons,Midsize-Large Station Wagons,Minicompact Cars,Minivan - 2WD,Minivan - 4WD,Small Pickup Trucks,Small Pickup Trucks 2WD,...,Standard Pickup Trucks 4WD,Standard Pickup Trucks/2wd,Standard Sport Utility Vehicle 2WD,Standard Sport Utility Vehicle 4WD,Subcompact Cars,Two Seaters,Vans,Vans Passenger,"Vans, Cargo Type","Vans, Passenger Type"
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49841,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
49842,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
49843,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
49844,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [24]:
# pick object/str unique types, doesn't make sense for numeric/continuous values
(
    df.select_dtypes(object).nunique().index
)

Index(['make', 'model', 'trany', 'drive', 'VClass', 'eng_dscr', 'evMotor',
       'fuelType', 'trans_dscr'],
      dtype='object')

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49846 entries, 0 to 49845
Data columns (total 19 columns):
 #   Column      Non-Null Count  Dtype                           
---  ------      --------------  -----                           
 0   year        49846 non-null  int64                           
 1   make        49846 non-null  object                          
 2   model       49846 non-null  object                          
 3   trany       49835 non-null  object                          
 4   drive       48660 non-null  object                          
 5   VClass      49846 non-null  object                          
 6   eng_dscr    31637 non-null  object                          
 7   barrels08   49846 non-null  float64                         
 8   city08      49846 non-null  int64                           
 9   comb08      49846 non-null  int64                           
 10  range       49846 non-null  int64                           
 11  evMotor     3447 non-null   

In [29]:
# Deploying one-hot encoding
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
disply_imputer = SimpleImputer(strategy='median')
std_scaler = StandardScaler()
cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest


cat_cols = ['make', 'model', 'trany', 'drive',
        'VClass', 'eng_dscr', 'evMotor', 'fuelType', 'trans_dscr']

# Encoding needs to happen in sequence (after or before the transformer steps)
# Why? because everything within a step is executed parallely (cyl_imputer and displ_imputer)
cat_pipeline = Pipeline([
    ('cat_imputer', cat_imputer),
    ('one_hot_encoder', one_hot_encoder)
])

preprocessor = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', disply_imputer, ['displ']),
        ('cat_pipeline', cat_pipeline, cat_cols)
    ],
    remainder='passthrough'
)

# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # if we print tmp_X now, function transformer will have stored the intermediate value of X between preprocessor and std_scaler in it
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name':'tmp_X'})), 
    ('std_scaler', std_scaler),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)


# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


R2 score is 0.9685591679801825
mse is 0.04727504673546944


c:\repos\ai-portfolio\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1, 5, 6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\repos\ai-portfolio\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1, 5, 6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [30]:
!pip install category_encoders

In [45]:
# Hash Encoding
high_cardinality_cols = (
    df.select_dtypes(object)
    .nunique()
    .pipe(lambda s: s[s>40]) # only pick high cardinality values i.e., columns with >40 unique values
    .index
).to_numpy()

low_cardinality_cols =  (
    df.select_dtypes(object)
    .nunique()
    .pipe(lambda s: s[s<40]) # only pick high cardinality values i.e., columns with >40 unique values
    .index
).to_numpy()


high_cardinality_cols=list(high_cardinality_cols)
low_cardinality_cols=list(low_cardinality_cols)

high_cardinality_cols



['make', 'model', 'eng_dscr', 'evMotor', 'trans_dscr']

In [ ]:
# Exploring hashencoding
from category_encoders import hashing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
disply_imputer = SimpleImputer(strategy='median')
std_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True) # for high cardinality values use a hash function to map the entire cardinality to a fixed set of cols (10 here)

preprocessor = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', disply_imputer, ['displ']),
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        ('hashing_encoder', hashing_encoder, high_cardinality_cols),
    ],
    remainder='drop'
)

# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # if we print tmp_X now, function transformer will have stored the intermediate value of X between preprocessor and std_scaler in it
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name':'tmp_X'})), 
    ('std_scaler', std_scaler),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)


# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


R2 score is 0.859361961340446
mse is 0.10430743853708524
